In [1]:
import os
import sys

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

# 가상환경 확인
print(f"Python 실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다")

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

Python 실행 경로: d:\yugyeong\hanwha_0902\days\ex_0918\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[ERROR] Google API 키 미설정
[OK] LangSmith 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [2]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        
        MessagesPlaceholder(variable_name = "chat_history"),
        
        ("human", "{input}"),
    ]
)

In [7]:
chain = prompt | llm

store = {}

In [8]:
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    
    return store[session_id]

In [15]:
conversation_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [11]:
response = conversation_chain.invoke(
    {
        "input": "안녕하세요? 만나서 반갑습니다. 제 이름은 테디입니다."
    },
    config={
        "configurable": {
            "session_id": "user-1"
        }
    }
)

print(response.content)

안녕하세요, 테디님! 만나서 반갑습니다. 저는 챗GPT예요. 오늘 무엇을 도와드릴까요?


In [12]:
response = conversation_chain.invoke(
    {
        "input": "제 이름이 뭐였죠?"
    },
    config={
        "configurable": {
            "session_id": "user-1"
        }
    }
)

print(response.content)

테디님이세요. 어떻게 도와드릴까요?


In [13]:
history = get_session_history("user-1")

print(history.messages)

[HumanMessage(content='안녕하세요? 만나서 반갑습니다. 제 이름은 테디입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 테디님! 만나서 반갑습니다. 저는 챗GPT예요. 오늘 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 31, 'total_tokens': 197, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ETGOWmLZmvNWh8lAoUjLK1NMAmztj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0eaa7-89f0-79e1-8142-c5e779186456-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 166, 'total_tokens': 197, 'inpu